In [ ]:
import os
import pandas as pd
import numpy as np

# =========================
# Config
# =========================
BASE_DIR = "./"
OUT_DIR  = "./statistics_description_csv"
OUT_PREFIX = "stats_desc__"

EXCLUDE_COLS = {
    "label", "target", "y",
    "SEED", "seed", "trial", "dataset_id"
}

# 출력 디렉토리 생성
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# CSV 자동 탐색
# =========================
CSV_FILES = [
    f for f in os.listdir(BASE_DIR)
    if f.endswith(".csv") and not f.startswith(OUT_PREFIX)
]

if not CSV_FILES:
    raise RuntimeError("No input CSV files found in ./")

print("[INFO] Input CSV files:")
for f in CSV_FILES:
    print(" -", f)

all_tables = []

# =========================
# Main loop
# =========================
for csv_name in CSV_FILES:
    df = pd.read_csv(os.path.join(BASE_DIR, csv_name))

    # index-like column 제거
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # 제외 컬럼 제거
    df = df.drop(
        columns=[c for c in df.columns if c in EXCLUDE_COLS],
        errors="ignore"
    )

    # 숫자 컬럼만
    num_df = df.select_dtypes(include=[np.number])

    if num_df.empty:
        print(f"[SKIP] {csv_name}: no numeric columns")
        continue

    # =========================
    # Descriptive statistics
    # =========================
    desc = pd.DataFrame({
        "Variable": num_df.columns,
        "N":      num_df.count().values,
        "Mean":   num_df.mean().values,
        "Std":    num_df.std().values,
        "Min":    num_df.min().values,
        "Q1":     num_df.quantile(0.25).values,
        "Median": num_df.median().values,
        "Q3":     num_df.quantile(0.75).values,
        "Max":    num_df.max().values,
    }).round(4)

    # 파일별 저장
    out_file = os.path.join(OUT_DIR, OUT_PREFIX + csv_name)
    desc.to_csv(out_file, index=False)

    print(f"[OK] Saved -> {out_file}")

    all_tables.append(desc)

# =========================
# ALL datasets 병합 저장
# =========================
if all_tables:
    merged = pd.concat(all_tables, ignore_index=True)
    merged_out = os.path.join(OUT_DIR, OUT_PREFIX + "ALL_DATASETS.csv")
    merged.to_csv(merged_out, index=False)
    print("[OK] Saved merged table ->", merged_out)
else:
    print("[WARN] No descriptive tables generated.")


[INFO] Input CSV files:
 - data_with_features_GES.csv
 - data_with_features_GOLEM.csv
 - data_with_features_NOTEARS.csv
 - data_with_features_PC.csv
 - training_data.csv
 - training_data_standardization.csv
[OK] Saved -> ./statistics_description_csv\stats_desc__data_with_features_GES.csv
[OK] Saved -> ./statistics_description_csv\stats_desc__data_with_features_GOLEM.csv
[OK] Saved -> ./statistics_description_csv\stats_desc__data_with_features_NOTEARS.csv
[OK] Saved -> ./statistics_description_csv\stats_desc__data_with_features_PC.csv
[OK] Saved -> ./statistics_description_csv\stats_desc__training_data.csv
[OK] Saved -> ./statistics_description_csv\stats_desc__training_data_standardization.csv
[OK] Saved merged table -> ./statistics_description_csv\stats_desc__ALL_DATASETS.csv
